[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/08_data_cleaning/08_5_Splitting_Extracting.ipynb)

# 08.5: Splitting and Extracting Text

Notebook 08.4 taught you to normalize a string column: strip whitespace, fix capitalization, replace known variants. That is the right approach when a column contains a single value that is just formatted inconsistently.

But some columns contain multiple distinct pieces of information packed into one string. The Titanic `name` column is a good example: it holds a social title, a first name, and a last name all at once. A car name like `"chevrolet chevelle malibu"` contains a manufacturer and a model name. To use any of those pieces in an analysis, you need to pull them apart first.

Two tools do this work: `str.split()` breaks a string at a delimiter, and `str.extract()` uses a pattern to pull out a specific portion. Let's load the data.

In [1]:
import pandas as pd
import seaborn as sns

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = ["survived", "pclass", "name", "sex", "age", "sibsp", "parch", "fare"]

df[["name"]].head(8)

,name
0,Mr. Owen Harris Braund
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...
2,Miss. Laina Heikkinen
3,Mrs. Jacques Heath (Lily May Peel) Futrelle
4,Mr. William Henry Allen
5,Mr. James Moran
6,Mr. Timothy J McCarthy
7,Master. Gosta Leonard Palsson


## Understanding the Name column

Before extracting anything, you need to understand the structure. Each name follows this pattern:

```
Title. FirstName MiddleName LastName
```

For example, `"Mr. Owen Harris Braund"` or `"Mrs. John Bradley (Florence Briggs Thayer) Cumings"`. The title is always the first word followed by a period. The period-plus-space is the delimiter that separates the title from the rest of the name.

## `str.split()` with `expand=True`

`str.split(delimiter)` breaks each string at every occurrence of the delimiter. The `expand=True` argument converts the result into separate columns of a DataFrame instead of a column of lists.

Splitting on `". "` (period-space) separates the title from everything that follows.

In [2]:
name_parts = df["name"].str.split(". ", expand=True, n=1)
name_parts.head(8)

,0,1
0,Mr,Owen Harris Braund
1,Mrs,John Bradley (Florence Briggs Thayer) Cumings
2,Miss,Laina Heikkinen
3,Mrs,Jacques Heath (Lily May Peel) Futrelle
4,Mr,William Henry Allen
5,Mr,James Moran
6,Mr,Timothy J McCarthy
7,Master,Gosta Leonard Palsson


Column `0` contains the title (`Mr`, `Mrs`, `Miss`, `Master`). Column `1` contains everything after the period and space. The `n=1` argument limits the split to at most one occurrence, which prevents the split from breaking on any other periods that might appear in the string (such as middle initials).

Assigning each part to a meaningful column name is one more step.

In [3]:
split_result = df["name"].str.split(". ", expand=True, n=1)
df["title"] = split_result[0]
df["rest"] = split_result[1]

df[["name", "title", "rest"]].head(5)

,name,title,rest
0,Mr. Owen Harris Braund,Mr,Owen Harris Braund
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,Mrs,John Bradley (Florence Briggs Thayer) Cumings
2,Miss. Laina Heikkinen,Miss,Laina Heikkinen
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,Mrs,Jacques Heath (Lily May Peel) Futrelle
4,Mr. William Henry Allen,Mr,William Henry Allen


The `title` column is clean. The `rest` column holds the first and last name (and any parenthetical notes), which we do not need to parse further for most analyses.

Let's verify the titles that appear in the data.

In [4]:
df["title"].value_counts()

title
Mr          513
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
th            1
Jonkheer      1
Name: count, dtype: int64

`Mr` is by far the most common title, followed by `Miss` and `Mrs`. The rarer titles, including `Rev`, `Dr`, `Col`, and `Major`, appear fewer than ten times each. Having the title in its own column makes it straightforward to group by title or filter to a specific subgroup.

## `str.get()`: indexing into list-valued columns

When you call `str.split()` without `expand=True`, the result is a Series of lists. You can then index into those lists with `str.get(n)`, which picks the nth element from every list. This is useful when you need only one piece from a split.

The MPG dataset is a natural fit: each car name contains the manufacturer followed by the model, separated by spaces.

In [5]:
mpg = sns.load_dataset("mpg")
print("Sample car names:")
print(mpg["name"].head(8).tolist())

# Extract only the manufacturer: the first word before any space
mpg["make"] = mpg["name"].str.split().str.get(0)
mpg[["name", "make"]].head(8)

Sample car names:
['chevrolet chevelle malibu', 'buick skylark 320', 'plymouth satellite', 'amc rebel sst', 'ford torino', 'ford galaxie 500', 'chevrolet impala', 'plymouth fury iii']


,name,make
0,chevrolet chevelle malibu,chevrolet
1,buick skylark 320,buick
2,plymouth satellite,plymouth
3,amc rebel sst,amc
4,ford torino,ford
5,ford galaxie 500,ford
6,chevrolet impala,chevrolet
7,plymouth fury iii,plymouth


`.str.split()` without a delimiter splits on any whitespace. `.str.get(0)` picks the first word. The result is a clean `make` column. Chaining these two calls in one line is a common pattern for extracting a single word from a text field.

In [6]:
# Now we can group by manufacturer
mpg_by_make = (
    mpg.groupby("make")["mpg"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "avg_mpg", "count": "n"})
    .query("n >= 5")
    .sort_values("avg_mpg", ascending=False)
    .round(1)
)
mpg_by_make.head(10)

,avg_mpg,n
make,,
vw,39.0,6
honda,33.8,13
renault,32.9,5
datsun,31.1,23
mazda,30.9,10
volkswagen,29.1,15
fiat,28.9,8
toyota,28.4,25
audi,26.7,7


The groupby by `make` was not possible before the split. Without it, the `name` column has a unique value for almost every row, so grouping on it would produce hundreds of single-row groups. The one-line extraction unlocked this analysis.

## `str.extract()`: pulling out a specific pattern

Splitting on a delimiter works when the structure is simple: the first word, or everything before a certain character. But some fields have a specific piece buried inside a longer string that a simple split cannot reach cleanly. For those cases, `str.extract()` uses a **capture group** to specify exactly what to pull out.

A capture group is written as parentheses `()` inside a pattern. `str.extract()` returns the text that matched whatever is inside those parentheses, and ignores everything else.

Going back to the Titanic names: the title is always a word followed by a period. The pattern `(\w+)\.` means "one or more word characters followed by a period; the parentheses capture the word characters."

In [7]:
# Use str.extract() to pull the title directly from the original name column
df["title_extracted"] = df["name"].str.extract(r'(\w+)\.')
df[["name", "title", "title_extracted"]].head(6)

,name,title,title_extracted
0,Mr. Owen Harris Braund,Mr,Mr
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,Mrs,Mrs
2,Miss. Laina Heikkinen,Miss,Miss
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,Mrs,Mrs
4,Mr. William Henry Allen,Mr,Mr
5,Mr. James Moran,Mr,Mr


Both the `str.split()` approach and `str.extract()` produce the same title column. The difference is in what you are specifying: `split` says "break at this delimiter and take piece number 0"; `extract` says "find this pattern anywhere in the string and return the captured portion."

`extract` is more flexible when the piece you want is not neatly positioned at the start or end, or when a simple delimiter does not exist.

## Named capture groups

When a pattern has multiple capture groups, `str.extract()` returns a DataFrame with one column per group. By default the columns are numbered 0, 1, 2. **Named groups** with `(?P<name>...)` give the output columns meaningful names automatically.

In [8]:
# Extract title and first name simultaneously from the original name column
name_parts_df = df["name"].str.extract(
    r'^(?P<title>\w+)\.\s+(?P<first>\w+)'
)
name_parts_df.head(8)

,title,first
0,Mr,Owen
1,Mrs,John
2,Miss,Laina
3,Mrs,Jacques
4,Mr,William
5,Mr,James
6,Mr,Timothy
7,Master,Gosta


The pattern `^(?P<title>\w+)\.\s+(?P<first>\w+)` reads as:
- `^`: start of the string
- `(?P<title>\w+)`: capture one or more word characters, label the group `title`
- `\.`: match a literal period
- `\s+`: match one or more spaces
- `(?P<first>\w+)`: capture the next word, label the group `first`

Both pieces are extracted in a single call. The resulting DataFrame already has the right column names, ready to assign back to the original DataFrame.

We now have a clean `title` column alongside the rest of the passenger data. Clean up the intermediate working columns, then compute survival rates grouped by title. To focus on titles with enough passengers to be meaningful, filter to groups with at least ten passengers.

In [9]:
# Clean up intermediate columns and show survival rates by title
df = df.drop(columns=["rest", "title_extracted"])

title_survival = (
    df.groupby("title")["survived"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "survival_rate", "count": "n"})
    .query("n >= 10")
    .sort_values("survival_rate", ascending=False)
    .round(2)
)
title_survival

,survival_rate,n
title,,
Mrs,0.79,125
Miss,0.70,182
Master,0.57,40
Mr,0.16,513


This analysis was not possible from the original dataset. After extracting the title, the pattern is clear: `Mrs` and `Miss` have survival rates near 75%, while `Mr` sits below 20%. `Master` (young boys) has a substantially higher rate than `Mr` (adult men), consistent with the "women and children first" evacuation policy.

All of this insight came from a string that originally looked like `"Mr. Owen Harris Braund"`, which is opaque to any groupby without the extraction step.

## What's next

You have now used `str.split()` to break on a delimiter and `str.extract()` to pull out a pattern. Both relied on relatively simple patterns: a period-space separator, a word followed by a period. Real-world data often has more complex patterns: phone numbers in multiple formats, currency symbols that vary by country, ZIP codes mixed with full addresses. Notebook 08.6 introduces regular expressions as a precise tool for those cleaning problems, building on the capture group syntax you just learned.